Scenario: AI Symptom Tracker (Question-Based)
1. State Definition
The assistant maintains a notebook-like state for each patient:
- symptom → The patient’s reported issue (e.g., "persistent cough").
- observations → Notes or snippets generated by Groq about possible causes or related conditions.
- analysis → A synthesized interpretation of the observations.
- recommendation → A simplified, non-medical summary suggesting next steps (e.g., "consult a doctor if symptoms persist").
- steps_done → A counter tracking progress through the workflow.

2. Workflow (Graph of States)
Each patient query flows through nodes:
- Symptom Input Node
- Patient reports a symptom.
- State updates: symptom = "persistent cough"
- Observation Node (Groq API)
- Groq generates possible related factors or general information.
- Updates observations.
- Analysis Node
- Joins observations and extracts key insights.
- Updates analysis.
- Conditional Node
- If fewer than 3 observations are collected → loop back to Observation Node.
- If 3+ observations are available → move to Recommendation Node.
- Recommendation Node
- Generates a simplified, non-medical recommendation (e.g., "Seek medical advice if cough lasts more than 2 weeks").
- Updates recommendation.
- End Node
- Delivers the final recommendation to the patient.

In [1]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List
import requests
from google.colab import userdata
import json

# 1. DEFINE STATE
class SymptomState(TypedDict):
    symptom: str
    observations: List[str]
    analysis: str
    recommendation: str
    steps_done: int


# 2. NODE: GENERATE OBSERVATION (Groq)
def generate_observation(state: SymptomState):
    groq_api_key = userdata.get('newapi')

    if not groq_api_key:
        raise ValueError("Groq API key not found.")

    prompt = f"""
    Symptom: {state['symptom']}

    Provide ONE general observation about possible causes or related conditions.
    Keep it informational, not diagnostic.

    Return ONLY JSON:
    {{
        "observation": "..."
    }}
    """

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [{"role": "user", "content": prompt}],
        }
    )

    if response.status_code != 200:
        raise Exception(f"Groq API error: {response.text}")

    data = response.json()
    content = data["choices"][0]["message"]["content"]

    try:
        result = json.loads(content)
    except:
        raise ValueError(f"Invalid JSON: {content}")

    new_observation = result["observation"]

    print(f"\nObservation {len(state['observations'])+1}: {new_observation}")

    return {
        "observations": state["observations"] + [new_observation],
        "steps_done": state["steps_done"] + 1
    }


# 3. NODE: ANALYSIS
def analyze_observations(state: SymptomState):
    combined = " ".join(state["observations"])

    analysis = f"Based on observations: {combined}"

    print("\nAnalysis Generated")

    return {"analysis": analysis}


# 4. NODE: RECOMMENDATION (Groq)
def generate_recommendation(state: SymptomState):
    groq_api_key = userdata.get('newapi')

    prompt = f"""
    Symptom: {state['symptom']}
    Analysis: {state['analysis']}

    Provide a simple, non-medical recommendation.
    Do NOT diagnose.

    Example: "Consult a doctor if symptoms persist"

    Return ONLY JSON:
    {{
        "recommendation": "..."
    }}
    """

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [{"role": "user", "content": prompt}],
        }
    )

    data = response.json()
    content = data["choices"][0]["message"]["content"]

    try:
        result = json.loads(content)
    except:
        raise ValueError(f"Invalid JSON: {content}")

    return {"recommendation": result["recommendation"]}


# 5. NODE: FINAL OUTPUT
def final_output(state: SymptomState):
    print("\nFINAL RESULT")
    print("Symptom:", state["symptom"])
    print("\nObservations:")
    for i, obs in enumerate(state["observations"], 1):
        print(f"{i}. {obs}")

    print("\nAnalysis:", state["analysis"])
    print("\nRecommendation:", state["recommendation"])

    return {}


# 6. CONDITIONAL FUNCTION
def check_observations(state: SymptomState):
    if len(state["observations"]) < 3:
        return "more_observations"
    else:
        return "enough_observations"


# 7. BUILD GRAPH
graph = StateGraph(SymptomState)

graph.add_node("observe", generate_observation)
graph.add_node("analyze", analyze_observations)
graph.add_node("recommend", generate_recommendation)
graph.add_node("final", final_output)

# Entry
graph.set_entry_point("observe")

# Conditional loop
graph.add_conditional_edges(
    "observe",
    check_observations,
    {
        "more_observations": "observe",
        "enough_observations": "analyze"
    }
)

# Remaining flow
graph.add_edge("analyze", "recommend")
graph.add_edge("recommend", "final")
graph.add_edge("final", END)

app = graph.compile()


# 8. RUN
if __name__ == "__main__":
    symptom = input("Enter your symptom: ")

    state = {
        "symptom": symptom,
        "observations": [],
        "analysis": "",
        "recommendation": "",
        "steps_done": 0
    }

    app.invoke(state)

Enter your symptom: Fever

Observation 1: Fever is often a symptom associated with infections, illnesses, and inflammatory conditions, and can also be caused by exposure to extreme temperatures or certain medications.

Observation 2: Fever can be associated with various conditions, including infections, illnesses, and medical emergencies, such as pneumonia, meningitis, and sepsis.

Observation 3: A fever can be a symptom of various infections, including viral and bacterial infections, as well as other medical conditions such as inflammation, autoimmune disorders, or even side effects of certain medications.

Analysis Generated

FINAL RESULT
Symptom: Fever

Observations:
1. Fever is often a symptom associated with infections, illnesses, and inflammatory conditions, and can also be caused by exposure to extreme temperatures or certain medications.
2. Fever can be associated with various conditions, including infections, illnesses, and medical emergencies, such as pneumonia, meningitis, a